<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_inference_t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [58]:
#!git clone "https://github.com/slomi23/ML_fx.git"
#!cd ML_fx/

In [59]:
!pip install wandb -q
!pip install neuralforecast pytorch-lightning wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [60]:
import pandas as pd
import wandb
from neuralforecast import NeuralForecast

run = wandb.init(
    project="ML_fx_DLinear_PyTorch",
    job_type="inference",
    name="dlinear_kaggle_submission_generation",
)

In [61]:
import os
import torch
import torch.nn as nn

class MovingAvg(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=0)

    def forward(self, x):
        front = x[:, 0:1].repeat(1, (self.kernel_size - 1) // 2)
        end = x[:, -1:].repeat(1, (self.kernel_size - 1) // 2)
        x_padded = torch.cat([front, x, end], dim=1)
        return self.avg(x_padded.unsqueeze(1)).squeeze(1)

class DLinear(nn.Module):
    def __init__(self, lookback=52, horizon=39, kernel_size=25):
        super().__init__()
        self.moving_avg = MovingAvg(kernel_size)
        self.linear_trend = nn.Linear(lookback, horizon)
        self.linear_seasonal = nn.Linear(lookback, horizon)

    def forward(self, x):
        trend = self.moving_avg(x)
        seasonal = x - trend
        out = self.linear_trend(trend) + self.linear_seasonal(seasonal)
        return out

print("Downloading model from W&B Registry...")
artifact = run.use_artifact("wandb-registry-model/Walmart-DLinear:latest", type="model")
artifact_dir = artifact.download()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK = 52
HORIZON = 39

model = DLinear(lookback=LOOKBACK, horizon=HORIZON, kernel_size=25).to(DEVICE)
model_path = os.path.join(artifact_dir, "best_dlinear_model.pth")
model.load_state_dict(torch.load(model_path, map_location=DEVICE))
model.eval()

print("DLinear Model successfully loaded into PyTorch!")

wandb:   1 of 1 files downloaded.  


DLinear Model successfully loaded into PyTorch!


In [62]:
import os
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
PROCCESSED_DATA_DIR = '/content/drive/MyDrive/'

train_df = pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, 'train_prepared.csv'))
test_df = pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, 'test_prepared.csv'))

train_df['Date'] = pd.to_datetime(train_df['Date'])
test_df['Date'] = pd.to_datetime(test_df['Date'])

train_sales = train_df.groupby(['Store', 'Dept']).apply(
    lambda g: g.sort_values('Date')['Weekly_Sales'].values,
    include_groups=False
).to_dict()

all_ids = []
all_preds = []

print("Generating non-repeating predictions on test set...")

model.eval()
with torch.no_grad():
    for (store, dept), group in test_df.groupby(['Store', 'Dept']):
        group = group.sort_values('Date')
        test_dates = group['Date'].dt.strftime('%Y-%m-%d').tolist()
        num_weeks = len(test_dates)

        store_dept_ids = [f"{store}_{dept}_{d}" for d in test_dates]

        history = train_sales.get((store, dept), np.array([]))
        if len(history) >= LOOKBACK:
            x = history[-LOOKBACK:].copy()
        else:
            x = np.pad(history, (max(0, LOOKBACK - len(history)), 0))

        scale = np.mean(np.abs(x))
        if scale == 0:
            scale = 1.0

        x_norm = torch.tensor(x / scale, dtype=torch.float32).unsqueeze(0).to(DEVICE)

        preds = model(x_norm).squeeze(0).cpu().numpy()
        preds_real = preds * scale

        series_preds = preds_real[:num_weeks]

        all_ids.extend(store_dept_ids)
        all_preds.extend(series_preds)

print(f"Total Prediction IDs Generated: {len(all_ids)}")
print(f"Total Predictions Generated:    {len(all_preds)}")

assert len(all_ids) == 115064, f"Expected 115064 rows, got {len(all_ids)}"
assert len(all_preds) == 115064, f"Expected 115064 rows, got {len(all_preds)}"

submission = pd.DataFrame({"Id": all_ids, "Weekly_Sales": all_preds})
submission["Weekly_Sales"] = submission["Weekly_Sales"].clip(lower=0).fillna(0)

submission_filename = "kaggle_submission_dlinear.csv"
submission.to_csv(submission_filename, index=False)

print(f"\nSubmission successfully saved to {submission_filename} with EXACTLY {len(submission)} rows!")
print("\nFirst 10 rows preview:")
print(submission.head(10))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generating non-repeating predictions on test set...
Total Prediction IDs Generated: 115064
Total Predictions Generated:    115064

Submission successfully saved to kaggle_submission_dlinear.csv with EXACTLY 115064 rows!

First 10 rows preview:
               Id  Weekly_Sales
0  1_1_2012-11-02  36631.116791
1  1_1_2012-11-09  21623.186285
2  1_1_2012-11-16  22363.352590
3  1_1_2012-11-23  21895.541217
4  1_1_2012-11-30  24324.038910
5  1_1_2012-12-07  30172.128127
6  1_1_2012-12-14  39178.040983
7  1_1_2012-12-21  43221.060600
8  1_1_2012-12-28  25692.262744
9  1_1_2013-01-04  19671.448019


In [63]:
sub_artifact = wandb.Artifact("kaggle_submission_dlinear", type="submission")
sub_artifact.add_file(submission_filename)
run.log_artifact(sub_artifact)

wandb.finish()